In [ ]:
#simple aGEN AI APP using lancganchain and openai

import os
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")
os.environ["LANGCHAIN_TRACE"] = "true"

In [ ]:
##data ingestion ---- from website we need to scrape the data 
from langchain_community.document_loaders import WebBaseLoader


In [ ]:
loader=WebBaseLoader("https://docs.smith.langchain.com/tutorials/Administrators/manage_spend")
loader

In [ ]:
docs = loader.load()
docs

In [ ]:
## load data ---> docs---->divide the data into chunks-->vectore embedding---> vector store
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents=text_splitter.split_documents(docs)
documents

In [ ]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings()

In [ ]:
from langchain_community.vectorstores import FAISS
vectorstoredb = FAISS.from_documents(documents, embeddings)
vectorstoredb

In [ ]:
#query from a vector db
query = "what are the steps to manage spend in smith?"
result= vectorstoredb.similarity_search(query)
result[0].page_content

In [ ]:
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model="gpt-4o")

In [ ]:
#retrival chain, documents chain

from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate   

prompt = ChatPromptTemplate.from_messages(
    """answer the question based on the following context:
    <context>
    {context}
    </context>
    """
)


document_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)
document_chain

In [ ]:
from langchain_core.documents import Document
document_chain.invoke({"input":" what are the steps to manage spend in smith?", 
                       "context":[Document(page_content=" Smith provides a comprehensive spend management solution that allows administrators to effectively control and optimize their organization's expenses. The steps to manage spend in Smith include:")]
                       })

In [ ]:
vectorstoredb

In [ ]:
##Retriver

retriver = vectorstoredb.as_retriever()
from langchain.chains import create_retrieval_chain
retrival_chain=create_retrieval_chain(retriever=retriver, documents_chain=document_chain)

In [ ]:
retrival_chain

In [ ]:
#get the responsefrom llm
response = retrival_chain.invoke({"input":" what are the steps to manage spend in smith?"})
response['answer']

In [ ]:
response